# Anotador TDAH · 01b · Backend `directo` + reparación de JSON

**Condición experimental derivada del cuaderno 01.** El backend es el mismo (`POST /api/generate` y extraer el JSON del texto), con una única diferencia: `extraer_json` **repara las comas finales** antes de parsear. Gemma (y otros modelos entrenados con código) emiten a veces JSON estilo JavaScript, con coma tras el último elemento de una lista válido en JS, inválido en JSON estricto.

**Qué mide este cuaderno:** la fracción de fallos de formato del backend directo que son *errores triviales de sintaxis*, recuperables con una reparación determinista. La distancia entre `directo` y `directo_reparado` en el cuaderno 05 es exactamente ese número.

**Importante para la comparabilidad:**
- No modificar el cuaderno 01: esta es una condición aparte, con su propio código de experimento (`directo_reparado-...`).
- Ejecutar con la **misma semana, repeticiones y temperatura** que el experimento del cuaderno 01 que se quiera comparar.

## 1 · Parámetros

In [ ]:
import datetime as dt
import json
import sqlite3
import time

import pandas as pd

# --- Parámetros del experimento (lo único que hay que tocar) ---
SEMANA       = 1            # semana de seguimiento (el dataset llega a la 24)
PACIENTES    = None         # None = todos los de la semana; o lista: ["P001", "P003"]
REPETICIONES = 3            # veces que se anota cada entrada
TEMPERATURA  = 0.7
MODELO       = "gemma4:e4b" # en Mercurio: gemma4:26b

# --- Rutas y conexión ---
RUTA_BD     = "datos/anotador.db"
OLLAMA_URL  = "http://127.0.0.1:11002"   # puerto del túnel a Ollama
INSTRUMENTO = "instrumentos/brief2.json"

BACKEND     = "directo_reparado"
EXPERIMENTO = f"{BACKEND}-s{SEMANA}-t{TEMPERATURA}-{dt.date.today():%Y%m%d}"
print(f"Código de experimento: {EXPERIMENTO}")

## 2 · Datos

Las entradas (texto libre de los padres) de la semana elegida, con el contexto del paciente.

In [ ]:
instrumento = json.load(open(INSTRUMENTO, encoding="utf-8"))
print(f"Instrumento: {instrumento['nombre']} ({len(instrumento['items'])} ítems)")

con = sqlite3.connect(RUTA_BD)

entradas = pd.read_sql(
    '''
    SELECT e.id_entrada, e.id_paciente, c.rol AS informante, e.fecha,
           p.fecha_nacimiento, p.sexo, e.texto
    FROM entrada e
    JOIN paciente p USING (id_paciente)
    JOIN cuidador c ON c.id_cuidador = e.id_cuidador
    JOIN referencia_sintetica r USING (id_entrada)
    WHERE r.semana = ?
    ORDER BY e.id_paciente
    ''',
    con, params=[SEMANA],
)
if PACIENTES:
    entradas = entradas[entradas["id_paciente"].isin(PACIENTES)]


def calcular_edad(nacimiento, observacion):
    nac = pd.to_datetime(nacimiento).date()
    obs = pd.to_datetime(observacion).date()
    return obs.year - nac.year - ((obs.month, obs.day) < (nac.month, nac.day))


entradas["edad"] = [
    calcular_edad(n, f) for n, f in zip(entradas["fecha_nacimiento"], entradas["fecha"])
]

print(f"Semana {SEMANA}: {len(entradas)} entradas de {entradas['id_paciente'].nunique()} pacientes")
entradas[["id_entrada", "id_paciente", "informante", "edad", "sexo", "texto"]].head()

## 3 · Prompts

Idénticos a los del cuaderno 01 (se construyen desde `brief2.json`).

In [ ]:
COMILLAS = '"' * 3  # delimitador del texto del padre dentro del prompt


def construir_prompt_sistema(instrumento):
    catalogo = "\n".join(
        f"  {it['id']}: [{it['escala']}] {it['texto']}" for it in instrumento["items"]
    )
    escalas = "\n".join(f"  - {e}: {d}" for e, d in instrumento["escalas"].items())
    n = instrumento["niveles_alerta"]
    return f'''Eres un {instrumento["rol_anotador"]}.

Tu tarea es analizar el texto libre de observación de un padre/madre sobre su hijo/a
y producir una anotación clínica estructurada en formato JSON, basada en el instrumento
{instrumento["nombre"]}.

## CATÁLOGO DE ÍTEMS ({len(instrumento["items"])} ítems)
{catalogo}

## ESCALAS
{escalas}

## NIVELES DE ALERTA
{" | ".join(n)}

## INSTRUCCIONES DE SALIDA
Responde ÚNICAMENTE con un objeto JSON válido, sin texto antes ni después, sin markdown.
Estructura requerida:
{{
  "items_detectados": [lista de números de ítem observables en el texto],
  "escalas_afectadas": [lista de escalas correspondientes],
  "nivel_alerta": "{n[0]}|{n[1]}|{n[2]}",
  "nota_clinica": "resumen clínico de 1-3 frases para el médico",
  "justificacion": "explicación del razonamiento (para auditoría)"
}}'''


def construir_prompt_usuario(e):
    return f'''## CONTEXTO DEL PACIENTE
- Edad: {e.edad} años
- Sexo: {e.sexo}
- Informante: {e.informante}

## TEXTO DEL PADRE/MADRE
{COMILLAS}{e.texto}{COMILLAS}

Analiza el texto y genera el JSON de anotación clínica.'''


prompt_sistema = construir_prompt_sistema(instrumento)
print(prompt_sistema[:400] + "\n[...]")

## 4 · El backend

La llamada al modelo es idéntica a la del cuaderno 01. Lo único que cambia es `extraer_json`: repara comas finales antes de parsear.

In [ ]:
import re

import requests


def extraer_json(texto):
    '''Extrae el primer JSON válido, reparando comas finales (variante REPARADA).

    Única reparación aplicada: eliminar la coma que precede a ] o } (JSON
    estilo JavaScript, frecuente en modelos entrenados con código). No se
    reparan otras variantes (comillas simples, claves sin comillas) a
    propósito: cada regla extra acercaría artificialmente esta condición a
    los backends estructurados y diluiría la comparación.

    Limitación conocida: la regex podría alterar una cadena que contuviera
    ",]" o ",}" literales dentro de un campo de texto (caso muy raro).
    '''
    if not texto:
        return None
    candidatos = [texto.strip()]
    m = re.search(r"\{.*\}", texto, re.DOTALL)
    if m:
        candidatos.append(m.group())
    limpio = re.sub(r"```(?:json)?", "", texto).strip()
    candidatos.append(limpio)
    for bruto in candidatos:
        arreglado = re.sub(r",(\s*[\]}])", r"\1", bruto)
        try:
            return json.loads(arreglado)
        except json.JSONDecodeError:
            continue
    return None

In [ ]:
def anotar(prompt_sistema, prompt_usuario):
    '''Llama al modelo y devuelve (anotacion | None, respuesta_cruda).'''
    payload = {
        "model": MODELO,
        "system": prompt_sistema,
        "prompt": prompt_usuario,
        "stream": False,
        "options": {"temperature": TEMPERATURA, "num_predict": 2048},
    }
    r = requests.post(f"{OLLAMA_URL}/api/generate", json=payload, timeout=180)
    r.raise_for_status()
    cruda = r.json()["response"]
    return extraer_json(cruda), cruda

## 5 · Una anotación de ejemplo

Antes de lanzar el experimento completo, una sola entrada para ver la anotación final.

In [ ]:
ejemplo = entradas.iloc[0]
print(f"Paciente {ejemplo.id_paciente} · {ejemplo.informante} · semana {SEMANA}")
print(f"Texto: {ejemplo.texto[:200]}...\n")

t0 = time.time()
anotacion, cruda = anotar(prompt_sistema, construir_prompt_usuario(ejemplo))
print(f"Latencia: {time.time() - t0:.1f}s\n")

if anotacion is None:
    print("[FALLO DE FORMATO] El modelo no devolvió un JSON válido:")
    print(cruda[:500])
else:
    print("ANOTACIÓN FINAL:")
    print(json.dumps(anotacion, indent=2, ensure_ascii=False))

## 6 · El experimento

Anota cada entrada de la semana `REPETICIONES` veces y guarda cada resultado en la tabla `experimento` con el código `EXPERIMENTO`.

> Si relanzas con el mismo código se añaden filas al mismo experimento. Para empezar de cero: `con.execute("DELETE FROM experimento WHERE codigo = ?", [EXPERIMENTO]); con.commit()`

In [ ]:
con.execute('''
CREATE TABLE IF NOT EXISTS experimento (
    id                INTEGER PRIMARY KEY,
    codigo            TEXT NOT NULL,      -- código del experimento (para comparar)
    creada_en         TEXT NOT NULL,
    backend           TEXT NOT NULL,
    modelo            TEXT NOT NULL,
    temperatura       REAL NOT NULL,
    semana            INTEGER,
    id_paciente       TEXT,
    id_entrada        INTEGER,
    repeticion        INTEGER,
    formato_ok        INTEGER,
    items_detectados  TEXT,               -- JSON: [int]
    escalas_afectadas TEXT,               -- JSON: [str]
    nivel_alerta      TEXT,
    nota_clinica      TEXT,
    justificacion     TEXT,
    latencia_s        REAL
)''')
con.commit()

total = len(entradas) * REPETICIONES
print(f"Experimento '{EXPERIMENTO}': {len(entradas)} entradas × {REPETICIONES} repeticiones "
      f"= {total} llamadas al modelo")

hechas = 0
for _, e in entradas.iterrows():
    prompt_usuario = construir_prompt_usuario(e)
    for rep in range(REPETICIONES):
        t0 = time.time()
        anotacion, cruda = anotar(prompt_sistema, prompt_usuario)
        latencia = time.time() - t0
        ok = anotacion is not None
        a = anotacion or {}
        con.execute(
            "INSERT INTO experimento (codigo, creada_en, backend, modelo, temperatura, "
            "semana, id_paciente, id_entrada, repeticion, formato_ok, items_detectados, "
            "escalas_afectadas, nivel_alerta, nota_clinica, justificacion, latencia_s) "
            "VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
            (EXPERIMENTO, dt.datetime.now().isoformat(timespec="seconds"), BACKEND,
             MODELO, TEMPERATURA, SEMANA, e.id_paciente, int(e.id_entrada), rep,
             int(ok), json.dumps(a.get("items_detectados", [])),
             json.dumps(a.get("escalas_afectadas", [])), a.get("nivel_alerta"),
             a.get("nota_clinica"), a.get("justificacion"), latencia),
        )
        con.commit()
        hechas += 1
        estado = "ok" if ok else "FALLO DE FORMATO"
        print(f"  [{hechas:>3}/{total}] {e.id_paciente} rep {rep + 1} → {estado} ({latencia:.1f}s)")

print(f"\nGuardado en la tabla `experimento` con codigo = '{EXPERIMENTO}'")

## 7 · Resultados

El dato clave de este cuaderno es `formato_ok`: compáralo con el del experimento gemelo del cuaderno 01 (en `05_comparacion_experimentos.ipynb`). La diferencia es la fracción de fallos triviales (comas finales) del backend directo.

In [ ]:
df = pd.read_sql(
    "SELECT * FROM experimento WHERE codigo = ?", con, params=[EXPERIMENTO]
)
print(f"{len(df)} anotaciones del experimento '{EXPERIMENTO}'\n")


def acuerdo_modal(niveles):
    '''Fracción de repeticiones que coincide con el nivel más frecuente.'''
    s = niveles.dropna()
    return round(s.value_counts().iloc[0] / len(s), 2) if len(s) else None


resumen = df.groupby("id_paciente").agg(
    repeticiones=("repeticion", "count"),
    formato_ok=("formato_ok", "mean"),
    acuerdo_nivel=("nivel_alerta", acuerdo_modal),
    latencia_media=("latencia_s", "mean"),
).round(2)

print(f"Formato válido: {df['formato_ok'].mean():.0%}")
print(f"Acuerdo medio del nivel de alerta entre repeticiones: {resumen['acuerdo_nivel'].mean():.2f}")
print(f"Latencia media por anotación: {df['latencia_s'].mean():.1f}s\n")
resumen

In [ ]:
# Las anotaciones de un paciente concreto, repetición a repetición
UN_PACIENTE = df["id_paciente"].iloc[0]   # cambiar por el que interese

detalle = df[df["id_paciente"] == UN_PACIENTE]
for _, fila in detalle.iterrows():
    print(f"— repetición {fila.repeticion}: nivel={fila.nivel_alerta} "
          f"items={fila.items_detectados}")
    print(f"  nota: {fila.nota_clinica}\n")

## 8 · Lectura para la tesis

> *"Una fracción X de los fallos de formato del backend directo corresponde a errores triviales de sintaxis (comas finales), recuperables con reparación determinista; los backends de salida estructurada (02–04) los eliminan por construcción."*

Donde X = `formato_ok(directo_reparado) − formato_ok(directo)` con los mismos parámetros.